# 11 — 1D CNN Proof-of-Concept

Tests whether a 1D Convolutional Neural Network can improve over the classical
classifiers (XGBoost, SVM, RF, LR) on the **polynomial coefficient representations**
(Chebyshev & Legendre, L2-normalised).

**Motivation:** The original article (Ambrosch et al. 2026) achieved its best results
with a CNN (ROC AUC 0.961) on XP coefficients. Polynomial coefficients have a
natural ordering by degree — low-degree coefficients capture coarse spectral shape,
higher degrees capture finer detail — making them a reasonable input for 1D
convolutions that learn local multi-scale patterns.

**Scope:** POC only — uses rep0 (5 folds) for speed.

**Improvements over v1:**
- **Optuna HPO** (20 trials per fold) over lr, weight_decay, noise, n_filters, dropout, n_layers, batch_size
- **Ensemble of 5 CNNs** (different seeds, averaged probabilities) to reduce variance
- **Cosine annealing** LR schedule
- **Gaussian noise augmentation** (magnitude tuned by HPO)

In [1]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import optuna
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    log_loss,
    roc_auc_score,
)
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

DATA_DIR = Path("data")
RESULTS_DIR = Path("results")

device = torch.device("cpu")
print(f"PyTorch {torch.__version__}, device: {device}")

ModuleNotFoundError: No module named 'torch'

## 1. Load splits and define representations

In [ ]:
with open(DATA_DIR / "splits_rskf.json") as f:
    all_splits = json.load(f)

splits = {k: v for k, v in all_splits.items() if k.startswith("rep0_")}
print(f"Using {len(splits)} folds (rep0 only)")

REPRESENTATIONS = []
for basis in ["chebyshev", "legendre"]:
    for n in [5, 10, 15, 20, 25, 30, 35, 40, 45, 50]:
        REPRESENTATIONS.append({
            "name": f"{basis}_{n}_L2",
            "file": f"{basis}_{n}_L2.csv",
            "n_features": n,
        })

print(f"\n{len(REPRESENTATIONS)} representations:")
for r in REPRESENTATIONS:
    print(f"  {r['name']:25s} ({r['n_features']:3d} features)")

Using 5 folds (rep0 only)

20 representations:
  chebyshev_5_L2            (  5 features)
  chebyshev_10_L2           ( 10 features)
  chebyshev_15_L2           ( 15 features)
  chebyshev_20_L2           ( 20 features)
  chebyshev_25_L2           ( 25 features)
  chebyshev_30_L2           ( 30 features)
  chebyshev_35_L2           ( 35 features)
  chebyshev_40_L2           ( 40 features)
  chebyshev_45_L2           ( 45 features)
  chebyshev_50_L2           ( 50 features)
  legendre_5_L2             (  5 features)
  legendre_10_L2            ( 10 features)
  legendre_15_L2            ( 15 features)
  legendre_20_L2            ( 20 features)
  legendre_25_L2            ( 25 features)
  legendre_30_L2            ( 30 features)
  legendre_35_L2            ( 35 features)
  legendre_40_L2            ( 40 features)
  legendre_45_L2            ( 45 features)
  legendre_50_L2            ( 50 features)


## 2. 1D CNN model (parameterised for HPO)

In [ ]:
class SpectralCNN(nn.Module):
    """1D CNN for coefficient-based binary classification.

    Architecture is parameterised for HPO: n_filters, dropout, n_layers.
    """

    def __init__(self, n_features: int, n_filters: int = 32,
                 dropout: float = 0.3, n_layers: int = 2):
        super().__init__()
        k1 = min(5, n_features)
        k2 = min(3, n_features)

        layers = [
            nn.Conv1d(1, n_filters, kernel_size=k1, padding=k1 // 2),
            nn.BatchNorm1d(n_filters),
            nn.ReLU(),
        ]
        in_ch = n_filters
        for _ in range(n_layers - 1):
            out_ch = in_ch * 2
            layers += [
                nn.Conv1d(in_ch, out_ch, kernel_size=k2, padding=k2 // 2),
                nn.BatchNorm1d(out_ch),
                nn.ReLU(),
            ]
            in_ch = out_ch

        layers.append(nn.AdaptiveAvgPool1d(1))
        self.features = nn.Sequential(*layers)

        self.head = nn.Sequential(
            nn.Linear(in_ch, in_ch // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(in_ch // 2, 1),
        )

    def forward(self, x):
        x = self.features(x)
        x = x.squeeze(-1)
        return self.head(x).squeeze(-1)


ENSEMBLE_SIZE = 5

for nf in [20, 50]:
    n_params = sum(p.numel() for p in SpectralCNN(nf).parameters())
    print(f"Default model (n_features={nf}): {n_params:,} params")
print(f"Ensemble size: {ENSEMBLE_SIZE}")

Default model (n_features=20): 8,705 params
Default model (n_features=50): 8,705 params
Ensemble size: 5


## 3. Evaluation helpers

In [ ]:
def pick_youden_threshold(y_true, y_prob, grid_size=200):
    thresholds = np.linspace(0, 1, grid_size)
    best_j, best_thr = -1, 0.5
    for thr in thresholds:
        y_pred = (y_prob >= thr).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
        sens = tp / (tp + fn) if (tp + fn) else 0
        spec = tn / (tn + fp) if (tn + fp) else 0
        j = sens + spec - 1
        if j > best_j:
            best_j, best_thr = j, thr
    return best_thr


def evaluate(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    sens = tp / (tp + fn) if (tp + fn) else 0.0
    spec = tn / (tn + fp) if (tn + fp) else 0.0
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    acc = (tp + tn) / (tp + tn + fp + fn)
    f1 = (2 * prec * sens) / (prec + sens) if (prec + sens) else 0.0
    return {
        "threshold": threshold,
        "sensitivity": sens,
        "specificity": spec,
        "precision": prec,
        "accuracy": acc,
        "f1": f1,
        "youden_j": sens + spec - 1.0,
        "roc_auc": roc_auc_score(y_true, y_prob),
        "pr_auc": average_precision_score(y_true, y_prob),
        "brier": brier_score_loss(y_true, y_prob),
        "log_loss": log_loss(y_true, y_prob),
    }

In [ ]:
def _stratified_val_split(X, y, val_frac, rng):
    """Split preserving class ratio."""
    idx = rng.permutation(len(X))
    pos = idx[y[idx] == 1]
    neg = idx[y[idx] == 0]
    n_val_pos = max(1, int(len(pos) * val_frac))
    n_val_neg = max(1, int(len(neg) * val_frac))
    val_idx = np.concatenate([pos[:n_val_pos], neg[:n_val_neg]])
    tr_idx = np.concatenate([pos[n_val_pos:], neg[n_val_neg:]])
    return X[tr_idx], y[tr_idx], X[val_idx], y[val_idx]


def to_tensor(X, y=None):
    t = torch.tensor(X, dtype=torch.float32).unsqueeze(1)
    if y is not None:
        return t, torch.tensor(y, dtype=torch.float32)
    return t


def train_single_model(
    X_tr, y_tr, X_val, y_val, n_features,
    lr=1e-3, weight_decay=1e-4, max_epochs=200, patience=20,
    batch_size=64, noise_std=0.0, n_filters=32, dropout=0.3, n_layers=2,
    seed=42,
):
    """Train one CNN with given hyperparameters. Returns (model, val_loss, stopped_epoch)."""
    torch.manual_seed(seed)
    X_tr_t, y_tr_t = to_tensor(X_tr, y_tr)
    X_val_t, y_val_t = to_tensor(X_val, y_val)

    n_pos = y_tr.sum()
    n_neg = len(y_tr) - n_pos
    pos_weight = torch.tensor([n_neg / max(n_pos, 1)], dtype=torch.float32)

    model = SpectralCNN(n_features, n_filters=n_filters, dropout=dropout, n_layers=n_layers)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_epochs)

    best_val_loss = float("inf")
    best_state = None
    epochs_no_improve = 0

    for epoch in range(max_epochs):
        model.train()
        perm = torch.randperm(len(X_tr_t))
        for i in range(0, len(X_tr_t), batch_size):
            batch_idx = perm[i : i + batch_size]
            xb, yb = X_tr_t[batch_idx], y_tr_t[batch_idx]
            if noise_std > 0:
                xb = xb + torch.randn_like(xb) * noise_std
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
        scheduler.step()

        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(X_val_t), y_val_t).item()

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                break

    model.load_state_dict(best_state)
    return model, best_val_loss, epoch + 1


def hpo_and_ensemble_fold(
    X_all, y_all, train_idx, test_idx, n_features,
    n_hpo_trials=20, ensemble_size=ENSEMBLE_SIZE,
):
    """Full pipeline for one fold: HPO -> train ensemble -> predict."""
    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_all[train_idx])
    X_te_scaled = scaler.transform(X_all[test_idx])
    y_tr = y_all[train_idx]
    y_te = y_all[test_idx]

    rng = np.random.RandomState(RANDOM_STATE)
    X_tr_inner, y_tr_inner, X_val, y_val = _stratified_val_split(
        X_tr_scaled, y_tr, val_frac=0.15, rng=rng,
    )

    # --- HPO with Optuna ---
    def objective(trial):
        params = {
            "lr": trial.suggest_float("lr", 1e-4, 1e-2, log=True),
            "weight_decay": trial.suggest_float("weight_decay", 1e-5, 1e-2, log=True),
            "noise_std": trial.suggest_float("noise_std", 0.0, 0.1),
            "n_filters": trial.suggest_categorical("n_filters", [16, 32, 64]),
            "dropout": trial.suggest_float("dropout", 0.1, 0.5),
            "n_layers": trial.suggest_int("n_layers", 1, 3),
            "batch_size": trial.suggest_categorical("batch_size", [32, 64, 128]),
        }
        _, val_loss, _ = train_single_model(
            X_tr_inner, y_tr_inner, X_val, y_val, n_features,
            max_epochs=150, patience=15, seed=RANDOM_STATE, **params,
        )
        return val_loss

    sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE)
    study = optuna.create_study(direction="minimize", sampler=sampler)
    study.optimize(objective, n_trials=n_hpo_trials, timeout=120)
    best_params = study.best_trial.params

    # --- Train ensemble with best params, varying seeds ---
    models = []
    for i in range(ensemble_size):
        model, _, _ = train_single_model(
            X_tr_inner, y_tr_inner, X_val, y_val, n_features,
            max_epochs=200, patience=20, seed=RANDOM_STATE + i + 1, **best_params,
        )
        models.append(model)

    # --- Ensemble prediction (average probabilities) ---
    X_val_t = to_tensor(X_val)
    X_te_t = to_tensor(X_te_scaled)

    val_probs_list, te_probs_list = [], []
    for model in models:
        model.eval()
        with torch.no_grad():
            val_probs_list.append(torch.sigmoid(model(X_val_t)).numpy())
            te_probs_list.append(torch.sigmoid(model(X_te_t)).numpy())

    val_probs = np.mean(val_probs_list, axis=0)
    te_probs = np.mean(te_probs_list, axis=0)

    thr = pick_youden_threshold(y_val, val_probs)
    metrics = evaluate(y_te, te_probs, thr)
    metrics["best_params"] = str(best_params)
    metrics["n_hpo_trials"] = n_hpo_trials
    return metrics

## 4. Run POC experiment

In [ ]:
CNN_RESULTS_PATH = RESULTS_DIR / "cnn_poc_results.csv"

completed = set()
if CNN_RESULTS_PATH.exists():
    df_prev = pd.read_csv(CNN_RESULTS_PATH)
    for _, row in df_prev.iterrows():
        completed.add((row["representation"], row["split"]))
    all_results = df_prev.to_dict("records")
    print(f"Resume: {len(completed)} cells already done, will skip them.")
else:
    all_results = []

for repr_cfg in REPRESENTATIONS:
    repr_file = DATA_DIR / repr_cfg["file"]
    if not repr_file.exists():
        print(f"SKIP — {repr_file.name} not found")
        continue

    pending = [
        s for s in splits.keys()
        if (repr_cfg["name"], s) not in completed
    ]
    if not pending:
        print(f"  {repr_cfg['name']:25s} — all folds done, skipping")
        continue

    df = pd.read_csv(repr_file)
    feat_cols = [c for c in df.columns if c not in ("source_id", "y")]
    X_all = df[feat_cols].to_numpy(dtype=np.float64)
    y_all = df["y"].to_numpy(dtype=int)

    print(f"\n{'─'*60}")
    print(f"  {repr_cfg['name']}  ({repr_cfg['n_features']} features)  [{len(pending)} folds pending]")
    print(f"{'─'*60}")

    for split_name in pending:
        split_idx = splits[split_name]
        torch.manual_seed(RANDOM_STATE)
        np.random.seed(RANDOM_STATE)

        train_idx = np.array(split_idx["train"])
        test_idx = np.array(split_idx["test"])

        metrics = hpo_and_ensemble_fold(
            X_all, y_all, train_idx, test_idx,
            n_features=repr_cfg["n_features"],
            n_hpo_trials=20,
            ensemble_size=ENSEMBLE_SIZE,
        )
        metrics["representation"] = repr_cfg["name"]
        metrics["n_features"] = repr_cfg["n_features"]
        metrics["split"] = split_name
        all_results.append(metrics)

        print(
            f"  {split_name}  ROC-AUC={metrics['roc_auc']:.4f}  "
            f"Sens={metrics['sensitivity']:.4f}  "
            f"Prec={metrics['precision']:.4f}"
        )

    # Save after each representation so progress is not lost
    pd.DataFrame(all_results).to_csv(CNN_RESULTS_PATH, index=False)

df_cnn = pd.DataFrame(all_results)
df_cnn.to_csv(CNN_RESULTS_PATH, index=False)
print(f"\nCollected {len(df_cnn)} results (saved to {CNN_RESULTS_PATH.name}).")

Resume: 100 cells already done, will skip them.
  chebyshev_5_L2            — all folds done, skipping
  chebyshev_10_L2           — all folds done, skipping
  chebyshev_15_L2           — all folds done, skipping
  chebyshev_20_L2           — all folds done, skipping
  chebyshev_25_L2           — all folds done, skipping
  chebyshev_30_L2           — all folds done, skipping
  chebyshev_35_L2           — all folds done, skipping
  chebyshev_40_L2           — all folds done, skipping
  chebyshev_45_L2           — all folds done, skipping
  chebyshev_50_L2           — all folds done, skipping
  legendre_5_L2             — all folds done, skipping
  legendre_10_L2            — all folds done, skipping
  legendre_15_L2            — all folds done, skipping
  legendre_20_L2            — all folds done, skipping
  legendre_25_L2            — all folds done, skipping
  legendre_30_L2            — all folds done, skipping
  legendre_35_L2            — all folds done, skipping
  legendre_40_L2 

## 5. CNN summary (mean across 5 folds)

In [ ]:
metric_cols = ["roc_auc", "pr_auc", "sensitivity", "specificity", "precision", "f1", "youden_j"]

cnn_summary = (
    df_cnn
    .groupby(["representation", "n_features"])[metric_cols]
    .agg(["mean", "std"])
)
cnn_summary.columns = [f"{col}_{stat}" for col, stat in cnn_summary.columns]
cnn_summary = cnn_summary.reset_index().sort_values("roc_auc_mean", ascending=False)

display_cols = ["representation", "n_features", "roc_auc_mean", "roc_auc_std",
                "sensitivity_mean", "precision_mean", "f1_mean"]
cnn_summary[display_cols].round(4)

,representation,n_features,roc_auc_mean,roc_auc_std,sensitivity_mean,precision_mean,f1_mean
9,chebyshev_5_L2,5,0.9198,0.0201,0.8261,0.7932,0.8070
0,chebyshev_10_L2,10,0.9170,0.0191,0.8351,0.7331,0.7760
19,legendre_5_L2,5,0.9148,0.0246,0.8333,0.8067,0.8171
1,chebyshev_15_L2,15,0.9145,0.0244,0.7832,0.8021,0.7879
8,chebyshev_50_L2,50,0.9126,0.0188,0.7884,0.7287,0.7522
13,legendre_25_L2,25,0.9115,0.0256,0.7921,0.7874,0.7871
4,chebyshev_30_L2,30,0.9089,0.0221,0.7920,0.7448,0.7655
14,legendre_30_L2,30,0.9087,0.0184,0.8297,0.6957,0.7541
2,chebyshev_20_L2,20,0.9084,0.0201,0.7885,0.6970,0.7364
15,legendre_35_L2,35,0.9077,0.0194,0.7957,0.7567,0.7708


In [ ]:
import matplotlib.pyplot as plt

# --- Fig 1: ROC-AUC by representation (CNN summary) ---
plot_df = cnn_summary.sort_values("roc_auc_mean", ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
colors = ["#2ca02c" if "legendre" in r else "#1f77b4" for r in plot_df["representation"]]
ax.barh(plot_df["representation"], plot_df["roc_auc_mean"], xerr=plot_df["roc_auc_std"],
        color=colors, edgecolor="white", capsize=3)
ax.set_xlabel("ROC-AUC (mean ± std, 5 folds)")
ax.set_title("1D CNN POC — ROC-AUC by Representation")
ax.set_xlim(0.88, 0.94)
ax.axvline(plot_df["roc_auc_mean"].max(), ls="--", color="grey", alpha=0.5, label="best CNN")
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "cnn_poc_roc_auc_by_repr.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Comparison with classical models

In [ ]:
focused = pd.read_csv(RESULTS_DIR / "focused_summary.csv")

poc_reprs = df_cnn["representation"].unique()
classical = focused[focused["representation"].isin(poc_reprs)].copy()

# Best classical model per representation
best_classical = (
    classical
    .sort_values("roc_auc_mean", ascending=False)
    .groupby("representation")
    .first()
    .reset_index()
)
best_classical["model"] = best_classical["classifier"]

# CNN summary with label
cnn_comp = cnn_summary.copy()
cnn_comp["model"] = "CNN_1D"

compare_cols = ["representation", "n_features", "model",
                "roc_auc_mean", "roc_auc_std",
                "sensitivity_mean", "precision_mean", "f1_mean"]

comparison = pd.concat([
    best_classical[compare_cols],
    cnn_comp[compare_cols],
]).sort_values(["representation", "model"])

print("CNN vs Best Classical Model per Representation")
print("=" * 70)
comparison.round(4)

CNN vs Best Classical Model per Representation


,representation,n_features,model,roc_auc_mean,roc_auc_std,sensitivity_mean,precision_mean,f1_mean
0,chebyshev_10_L2,10,CNN_1D,0.9170,0.0191,0.8351,0.7331,0.7760
0,chebyshev_10_L2,10,SVM_RBF,0.9241,0.0020,0.8332,0.7638,0.7960
1,chebyshev_15_L2,15,CNN_1D,0.9145,0.0244,0.7832,0.8021,0.7879
1,chebyshev_15_L2,15,SVM_RBF,0.9194,0.0034,0.8330,0.7473,0.7866
2,chebyshev_20_L2,20,CNN_1D,0.9084,0.0201,0.7885,0.6970,0.7364
2,chebyshev_20_L2,20,LogisticRegression,0.9164,0.0021,0.8328,0.7828,0.8063
3,chebyshev_25_L2,25,CNN_1D,0.9075,0.0215,0.7849,0.7339,0.7503
3,chebyshev_25_L2,25,SVM_RBF,0.9189,0.0027,0.8374,0.7319,0.7803
4,chebyshev_30_L2,30,CNN_1D,0.9089,0.0221,0.7920,0.7448,0.7655
4,chebyshev_30_L2,30,SVM_RBF,0.9191,0.0028,0.8378,0.7262,0.7773


In [ ]:
# --- Fig 2: CNN vs Best Classical — ROC-AUC side by side ---
# numpy already imported above

matched_reprs = sorted(df_delta["representation"].unique())
cnn_vals = []
cls_vals = []
cls_labels = []
for r in matched_reprs:
    c_row = cnn_comp[cnn_comp["representation"] == r].iloc[0]
    b_row = best_classical[best_classical["representation"] == r].iloc[0]
    cnn_vals.append(c_row["roc_auc_mean"])
    cls_vals.append(b_row["roc_auc_mean"])
    cls_labels.append(b_row["model"])

x = np.arange(len(matched_reprs))
w = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - w/2, cls_vals, w, label="Best Classical", color="#ff7f0e", edgecolor="white")
ax.bar(x + w/2, cnn_vals, w, label="CNN (1D)", color="#1f77b4", edgecolor="white")
ax.set_xticks(x)
ax.set_xticklabels([f"{r}\n({cl})" for r, cl in zip(matched_reprs, cls_labels)],
                    rotation=30, ha="right", fontsize=8)
ax.set_ylabel("ROC-AUC (mean)")
ax.set_title("CNN vs Best Classical Model per Representation")
ax.set_ylim(0.88, 0.94)
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "cnn_poc_vs_classical.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Delta table: CNN minus best classical
delta_rows = []
for repr_name in poc_reprs:
    cnn_match = cnn_comp[cnn_comp["representation"] == repr_name]
    cls_match = best_classical[best_classical["representation"] == repr_name]
    if cnn_match.empty or cls_match.empty:
        continue
    cnn_row = cnn_match.iloc[0]
    cls_row = cls_match.iloc[0]
    delta_rows.append({
        "representation": repr_name,
        "best_classical": cls_row["model"],
        "delta_roc_auc": cnn_row["roc_auc_mean"] - cls_row["roc_auc_mean"],
        "delta_sensitivity": cnn_row["sensitivity_mean"] - cls_row["sensitivity_mean"],
        "delta_precision": cnn_row["precision_mean"] - cls_row["precision_mean"],
        "delta_f1": cnn_row["f1_mean"] - cls_row["f1_mean"],
    })

df_delta = pd.DataFrame(delta_rows)
print("Delta: CNN − Best Classical (positive = CNN wins)")
print(f"({len(delta_rows)}/{len(poc_reprs)} representations matched)")
print("=" * 60)
df_delta.round(4)

Delta: CNN − Best Classical (positive = CNN wins)
(5/8 representations matched)


,representation,best_classical,delta_roc_auc,delta_sensitivity,delta_precision,delta_f1
0,chebyshev_5_L2,SVM_RBF,-0.0075,-0.0074,0.0017,-0.0042
1,chebyshev_10_L2,SVM_RBF,-0.0071,0.0020,-0.0307,-0.0200
2,chebyshev_20_L2,LogisticRegression,-0.0079,-0.0442,-0.0857,-0.0699
3,chebyshev_50_L2,LogisticRegression,-0.0048,-0.0452,-0.0499,-0.0521
4,legendre_50_L2,LogisticRegression,-0.0125,-0.0320,-0.1013,-0.0750


In [ ]:
# --- Fig 3: Delta ROC-AUC (CNN − Classical) ---
delta_sorted = df_delta.sort_values("delta_roc_auc")

fig, ax = plt.subplots(figsize=(9, 4))
colors = ["#d62728" if d < 0 else "#2ca02c" for d in delta_sorted["delta_roc_auc"]]
ax.barh(delta_sorted["representation"], delta_sorted["delta_roc_auc"], color=colors, edgecolor="white")
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("Δ ROC-AUC (CNN − Best Classical)")
ax.set_title("CNN Improvement over Classical Models (negative = classical wins)")

for i, (_, row) in enumerate(delta_sorted.iterrows()):
    ax.text(row["delta_roc_auc"] - 0.001, i, f'{row["delta_roc_auc"]:+.4f}',
            va="center", ha="right" if row["delta_roc_auc"] < 0 else "left",
            fontsize=9, color="white", fontweight="bold")

plt.tight_layout()
plt.savefig(RESULTS_DIR / "cnn_poc_delta_roc_auc.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# --- Fig 4: Multi-metric comparison heatmap ---
metrics_to_show = ["roc_auc_mean", "pr_auc_mean", "sensitivity_mean", "precision_mean", "f1_mean"]
metric_labels = ["ROC-AUC", "PR-AUC", "Sensitivity", "Precision", "F1"]

heat_data = []
for r in matched_reprs:
    c_row = cnn_comp[cnn_comp["representation"] == r].iloc[0]
    b_row = best_classical[best_classical["representation"] == r].iloc[0]
    for m, ml in zip(metrics_to_show, metric_labels):
        heat_data.append({"representation": r, "metric": ml,
                          "CNN": c_row[m], "Classical": b_row[m],
                          "delta": c_row[m] - b_row[m]})

heat_df = pd.DataFrame(heat_data)
pivot = heat_df.pivot(index="representation", columns="metric", values="delta")[metric_labels]

fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(pivot.values, cmap="RdYlGn", aspect="auto", vmin=-0.06, vmax=0.06)
ax.set_xticks(range(len(metric_labels)))
ax.set_xticklabels(metric_labels, fontsize=9)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index, fontsize=9)
for i in range(len(pivot.index)):
    for j in range(len(metric_labels)):
        ax.text(j, i, f"{pivot.values[i, j]:+.3f}", ha="center", va="center", fontsize=8)
plt.colorbar(im, ax=ax, label="Δ (CNN − Classical)")
ax.set_title("CNN vs Classical — Metric Deltas by Representation")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "cnn_poc_metric_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Verdict

In [ ]:
avg_delta = df_delta["delta_roc_auc"].mean()
wins = (df_delta["delta_roc_auc"] > 0).sum()
total = len(df_delta)

print(f"Average ROC AUC delta (CNN - classical): {avg_delta:+.4f}")
print(f"CNN wins on {wins}/{total} representations")
print()
if wins > total / 2 and avg_delta > 0:
    print("VERDICT: CNN shows promise. Worth adding to the full experiment.")
elif wins > 0:
    print("VERDICT: Mixed results. CNN helps on some representations but not all.")
    print("Consider running full 10x5 CV before deciding.")
else:
    print("VERDICT: CNN does not improve over classical models on polynomial features.")
    print("The sequential structure may not provide enough benefit at these dimensions.")

Average ROC AUC delta (CNN - classical): -0.0080
CNN wins on 0/5 representations

VERDICT: CNN does not improve over classical models on polynomial features.
The sequential structure may not provide enough benefit at these dimensions.


In [ ]:
df_cnn.to_csv(RESULTS_DIR / "cnn_poc_results.csv", index=False)